# 데이터구조 04주차 실습: 스택(Stack)과 큐(Queue)

**주제:** Stack, Queue 클래스 구현 및 활용  
**학습 목표**
- LIFO/FIFO 원리와 ADT 인터페이스를 설명한다.
- Python으로 Stack과 Queue 클래스를 설계·구현·검증한다.
- 괄호 검사와 후위 표기식 계산에 Stack을 적용한다.
- 리스트와 `collections.deque`의 큐 구현 비용을 비교한다.

> 실행 환경: Python 3.8+ / Jupyter Notebook 또는 Google Colab


## 0. 준비
이번 실습에서는 내부 표현을 숨기고, 정해진 메서드로만 자료구조를 사용합니다. 빈 구조에서 삭제·조회할 때는 `IndexError`를 발생시키겠습니다.

In [9]:
from collections import deque

# 테스트 도우미: 오류가 예상되는 코드가 실제로 오류를 내는지 확인합니다.
def expect_error(error_type, func):
    try:
        func()
    except error_type:
        print(f"OK: {error_type.__name__} 발생")
    else:
        raise AssertionError(f"{error_type.__name__}가 발생해야 합니다.")

## 1. Stack ADT — LIFO
**Last-In, First-Out:** 가장 나중에 넣은 항목이 먼저 나온다.

구현할 인터페이스: `push`, `pop`, `peek`, `is_empty`, `size`

In [10]:
class Stack:
    def __init__(self):
        self._items = []

    def push(self, item):
        self._items.append(item)

    def pop(self):
        if self.is_empty():
            raise IndexError('빈 스택에서 pop할 수 없습니다.')
        return self._items.pop()

    def peek(self):
        if self.is_empty():
            raise IndexError('빈 스택에서 peek할 수 없습니다.')
        return self._items[-1]

    def is_empty(self):
        return len(self._items) == 0

    def size(self):
        return len(self._items)

In [11]:
# Stack 기본 테스트: TODO를 모두 구현한 뒤 실행하세요.
s = Stack()
assert s.is_empty() is True
s.push('A'); s.push('B'); s.push('C')
assert s.size() == 3
assert s.peek() == 'C'
assert s.pop() == 'C'
assert s.pop() == 'B'
assert s.pop() == 'A'
assert s.is_empty() is True
expect_error(IndexError, s.pop)
print('Stack 기본 테스트 통과!')

OK: IndexError 발생
Stack 기본 테스트 통과!


### 연습 1 — 괄호 검사
여는 괄호 `(`, `[`, `{`는 push하고, 닫는 괄호가 나오면 대응하는 여는 괄호가 top에 있는지 확인합니다.

- 닫는 괄호에서 스택이 비었거나 짝이 맞지 않으면 `False`
- 모든 문자를 처리한 뒤 스택이 비어 있으면 `True`

In [12]:
def is_balanced(expression):
    pairs = {')': '(', ']': '[', '}': '{'}
    stack = Stack()

    for ch in expression:
        if ch in pairs.values():
            stack.push(ch)
        elif ch in pairs:
            if stack.is_empty() or stack.pop() != pairs[ch]:
                return False

    return stack.is_empty()

# 기대 결과
assert is_balanced('(a + b) * [c - d]') is True
assert is_balanced('([)]') is False
assert is_balanced('(()') is False
assert is_balanced('') is True
print('괄호 검사 테스트 통과!')

괄호 검사 테스트 통과!


### 도전 2 — 후위 표기식(Postfix) 계산
공백으로 분리된 토큰을 입력받아 계산하세요. 예: `"2 3 + 4 *"` → `(2+3)*4` → `20`

규칙: 숫자는 push, 연산자는 pop 두 번 후 계산 결과를 push합니다. 두 번째로 pop한 값이 **왼쪽 피연산자**입니다.

In [13]:
def evaluate_postfix(expression):
    stack = Stack()
    operators = {'+', '-', '*', '/'}

    for token in expression.split():
        if token not in operators:
            stack.push(float(token))
            continue

        if stack.size() < 2:
            raise ValueError('피연산자가 부족한 후위 표기식입니다.')
        right = stack.pop()
        left = stack.pop()

        if token == '+':
            result = left + right
        elif token == '-':
            result = left - right
        elif token == '*':
            result = left * right
        else:
            result = left / right
        stack.push(result)

    if stack.size() != 1:
        raise ValueError('올바른 후위 표기식이 아닙니다.')
    return stack.pop()

assert evaluate_postfix('2 3 + 4 *') == 20
assert evaluate_postfix('10 2 / 3 -') == 2
print('후위 표기식 테스트 통과!')

후위 표기식 테스트 통과!


## 2. Queue ADT — FIFO
**First-In, First-Out:** 먼저 넣은 항목이 먼저 나온다.

큐의 앞(front)에서 제거하고, 뒤(rear)에 삽입합니다. Python 리스트의 `pop(0)`은 앞 원소를 꺼낸 뒤 나머지를 이동시켜 **O(n)**입니다. 실무에서는 `deque.popleft()`를 사용합니다.

In [14]:
class Queue:
    def __init__(self):
        self._items = deque()

    def enqueue(self, item):
        self._items.append(item)

    def dequeue(self):
        if self.is_empty():
            raise IndexError('빈 큐에서 dequeue할 수 없습니다.')
        return self._items.popleft()

    def front(self):
        if self.is_empty():
            raise IndexError('빈 큐에서 front를 조회할 수 없습니다.')
        return self._items[0]

    def is_empty(self):
        return len(self._items) == 0

    def size(self):
        return len(self._items)

In [15]:
# Queue 기본 테스트
q = Queue()
assert q.is_empty() is True
q.enqueue('A'); q.enqueue('B'); q.enqueue('C')
assert q.front() == 'A'
assert q.dequeue() == 'A'
assert q.dequeue() == 'B'
assert q.size() == 1
assert q.dequeue() == 'C'
expect_error(IndexError, q.front)
print('Queue 기본 테스트 통과!')

OK: IndexError 발생
Queue 기본 테스트 통과!


## 3. 응용 — 작업 대기열
아래 `process_jobs`는 들어온 작업을 순서대로 처리한 결과를 반환해야 합니다.

In [16]:
def process_jobs(jobs):
    queue = Queue()
    for job in jobs:
        queue.enqueue(job)

    processed = []
    while not queue.is_empty():
        processed.append(queue.dequeue())
    return processed

assert process_jobs(['print-A', 'print-B', 'print-C']) == ['print-A', 'print-B', 'print-C']
print('작업 대기열 테스트 통과!')

작업 대기열 테스트 통과!


## 4. 복잡도 정리 및 제출 체크

|구조|삽입|제거/조회|대표 구현|
|---|---:|---:|---|
|Stack|push O(1)|pop·peek O(1)|`list` 오른쪽 끝|
|Queue|enqueue O(1)|dequeue O(1)|`deque`|

**제출 전 체크**
1. 모든 TODO를 해결하고 모든 `assert`를 통과했는가?
2. 빈 Stack/Queue에서의 예외 처리를 했는가?
3. 각 메서드의 시간복잡도를 설명할 수 있는가?
4. 파일명을 `Week04_학번_이름.ipynb`로 하여 제출한다.
